# Session 2 — context sweep

**Question:** how far does context length scale before output quality or memory gives out?

All logic lives in `wca.sweep`, not in these cells. That is deliberate — a Colab
notebook is a *copy*, so edits pushed to the repo never reach an open tab, and cells
silently go stale. `pip install --upgrade` does reach the package. Keeping the cells
nearly empty means there is nothing here to go stale.

**Runtime → Change runtime type → T4 GPU, then Runtime → Restart session.**

In [ ]:
#@title 1. Install — must print 0.4.1 or higher
import sys
assert "wca" not in sys.modules, "Stale module. Runtime > Restart session, then re-run."

REPO = "https://github.com/quinyang/whole_codebase_auditor"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}

!pip install -q --upgrade --force-reinstall --no-deps "wca @ git+{REPO}@{BRANCH}"
!pip install -q "wca[gpu] @ git+{REPO}@{BRANCH}"

import wca
from wca.sweep import run_demo, run_sweep
print("wca", wca.__version__)
assert wca.__version__ >= "0.3.1", f"got {wca.__version__}; the push did not land"
print("OK")

In [ ]:
#@title 2. Sanity check — toy fixture, scored against ground truth (~3 min)
# Confirms the setup works before spending 30+ minutes on a sweep.
from wca.infer import MambaAuditor

auditor = MambaAuditor()          # prints its VRAM footprint; ~6 GiB means 4-bit worked
row = run_demo(auditor)

In [ ]:
#@title 3. The sweep — this is the session's deliverable (~30-60 min)
TARGET = "pallets/click"  #@param {type:"string"}

rows = run_sweep(TARGET, auditor=auditor, budgets=(2000, 4000, 8000, 16000, 24000))

In [ ]:
#@title 4. Save results + plot the two curves
import json, os
os.makedirs("/content/wca_runs", exist_ok=True)

with open("/content/wca_runs/sweep.json", "w") as fh:
    json.dump([r.to_dict() for r in rows], fh, indent=2)
for r in rows:
    if r.raw_output:
        open(f"/content/wca_runs/raw_{r.budget}.txt", "w").write(r.raw_output)

ok = [r for r in rows if not r.error and r.prompt_tokens]
if len(ok) >= 2:
    import matplotlib.pyplot as plt
    x = [r.prompt_tokens for r in ok]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
    a1.plot(x, [r.peak_gib for r in ok], "o-")
    a1.set(xlabel="context (tokens)", ylabel="peak VRAM (GiB)",
           title="Memory vs context\n(linear slope = eager path)")
    a1.grid(alpha=.3)
    a2.plot(x, [r.grounding_rate for r in ok], "o-", label="grounding rate")
    a2.plot(x, [1.0 if r.json_valid else 0.0 for r in ok], "s--", label="JSON valid")
    a2.set(xlabel="context (tokens)", ylabel="rate", ylim=(-0.05, 1.05),
           title="Output quality vs context")
    a2.legend(); a2.grid(alpha=.3)
    plt.tight_layout()
    plt.savefig("/content/wca_runs/sweep.png", dpi=140)
    plt.show()
    print("saved -> /content/wca_runs/sweep.png  (download it; this is the artifact)")
else:
    print("need >=2 successful budgets to plot")

## What each outcome means

| Pattern in the table | Reading | Next |
|---|---|---|
| `json` yes throughout, grounding rate flat | Model holds up at every budget tested | Push budgets past 24k |
| `json` yes → NO at some budget | Instruction-following decays as SSM state fills | Shorten the schema; move instructions *after* the code |
| grounding rate falls with context | Recall of exact detail decays — **the headline result** | Report it; this is the SSM trade-off, measured |
| OOM at a budget | Memory ceiling on the eager path | Note it; try `pip install kernels` |
| findings drop to 0 at high budget | Signal lost in the noise | Compare against the packer's `omitted` count |

**Every one of these is a result.** The sweep failing at 24k is as publishable as it
succeeding — it's a measurement of where a 7B SSM stops being usable on a T4.

Download `sweep.json`, `sweep.png` and the `raw_*.txt` files. Those are session 3's input.